# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip install -q schedule pytest # установка библиотек, если ещё не


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def get_book_data(book_url: str) -> dict:
   """
   Извлечение данных одной книги

   Берет адрес страницы (book_url)
   
   Возвращает словарь с информацией о книге:
      'title' - название
      'price' - цена
      'rating' - рейтинг
      'availability' - наличие
      'description' - описание
      'product_information' - полная информация
   """

   # НАЧАЛО ВАШЕГО РЕШЕНИЯ
   response = requests.get(book_url)
   response.raise_for_status()

   soup = BeautifulSoup(response.text, 'html.parser')

   title_tag = soup.find('h1')
   title = title_tag.text.strip() if title_tag else ''
   price_tag = soup.find('p', class_='price_color')
   price = price_tag.text.strip() if price_tag else ''

   rating_tag = soup.find('p', class_='star-rating')
   rating = 0
   if rating_tag:
      rating_class = rating_tag['class'][1]
      rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
      rating = rating_map.get(rating_class, 0)

   availability_tag = soup.find('p', class_='availability')
   availability = availability_tag.text.strip() if availability_tag else ''

   description = ''
   description_header = soup.find('div', id='product_description')
   if description_header:
      description_tag = description_header.find_next_sibling('p')
      if description_tag:
         description = description_tag.text.strip()

   product_info = {}
   table = soup.find('table', class_='table table-striped')
   if table:
      rows = table.find_all('tr')
      for row in rows:
         th = row.find('th')
         td = row.find('td')
         if th and td:
            product_info[th.text.strip()] = td.text.strip()

   book_data = {
      'title': title,
      'price': price,
      'rating': rating,
      'availability': availability,
      'description': description,
      'product_information': product_info
   }

   return book_data

   # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [10]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 'Â£51.77',
 'rating': 3,
 'available': '',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you u

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [8]:
def scrape_books(is_save: bool = False, max_workers: int = 20):
    """
    Парсинг всех страниц каталога
    
    Берет флаг сохранения информации (is_save) и 
    количество потоков для параллельных запросов (max_workers), нужно для ускорения
    
    Возвращает список словарей
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    catalogue_base = 'http://books.toscrape.com/catalogue/'
    page_template = catalogue_base + 'page-{}.html'
    total_pages = 50
    session = requests.Session()

    def fetch_page_links(page: int) -> list[str]:
        response = session.get(page_template.format(page))
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        return [catalogue_base + a['href']
                for a in soup.select('article.product_pod h3 a')]

    book_urls = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for links in pool.map(fetch_page_links, range(1, total_pages + 1)):
            book_urls.extend(links)

    def fetch_book(url: str) -> dict:
        return get_book_data(url)

    all_books = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(fetch_book, url) for url in book_urls]
        for future in as_completed(futures):
            all_books.append(future.result())

    if is_save:
        with open('books_data.txt', 'w', encoding='utf-8') as f:
            for book in all_books:
                f.write(str(book) + '\n')

    return all_books

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [9]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def run_scheduler(run_time):
    schedule.every().day.at(run_time).do(scrape_books, is_save=True)
    print(f'выполнит в {run_time}.')

    try:
        while True:
            schedule.run_pending()
            time.sleep(30)
    except KeyboardInterrupt:
        print('стоп')


run_scheduler('05:09')
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

выполнит в 05:09.
стоп


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.13.3, pytest-7.1.2, pluggy-1.6.0
rootdir: c:\Users\Admin\Desktop\Учеба\Python ДЗ\ДЗ3
plugins: anyio-4.9.0
collected 3 items

test\test_scraper.py ...                                                 [100%]

============================== warnings summary ===============================
..\..\..\..\AppData\Roaming\Python\Python313\site-packages\_pytest\assertion\rewrite.py:962: 15 warnings
  C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\_pytest\assertion\rewrite.py:962: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
    inlocs = ast.Compare(ast.Str(name.id), [ast.In()], [locs])

..\..\..\..\AppData\Roaming\Python\Python313\site-packages\_pytest\assertion\rewrite.py:965: 14 warnings
  C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\_pytest\assertion\rewrite.py:965: DeprecationWarning: ast.Str is dep

## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: сделал всё и всё получилось`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Проблемы начались ещё в самом начале, "! pip install -q schedule pytest # установка библиотек, если ещё не" данная команда срабатывала с ошибкой из-за комментария; также много времени пришлось провозиться со 2 заданием, так как его пришшлось оптимизировать, по коду, который я написал изначально, результат расчитывался за 5-7 минут, через многопоточность смогу ускорить до половины минуты`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовал ИИ, чтобы узнать причину ошибки данной команды "! pip install -q schedule pytest # установка библиотек, если ещё не", в промпт закинул команду и ошибку, которая выскочила, ИИ сразу дал ответ, что проблема в комментарии сразу после команда, в остальном использовал поисковик и ноутбуки с семинаров`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: всё понятно, если и были непонятки, уже исправил их, пока решал проблемы с кодом`

**5. Самооценка по критериям задания**:

`Ответ: 7-10`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: сложным оказалось для меня второе задание, так как пришлось гуглить способы оптимизировать парсинг`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

